# Iterators and Generators

The mechanics behind how Python (and PyTorch's `Dataset`/`DataLoader` on top of it) streams data: the iterator protocol, and generators as a lazy shortcut for writing iterators.

## Iterables: `__iter__` vs `__getitem__`

- Sequences (`list`, `tuple`, `str`, `range`) are iterables, but you don't manually track a position index yourself — `iter(seq)` returns an **iterator** object that tracks position internally, and each `next()` call advances it.
- An object is iterable via one of two protocols:
  - **`__iter__`** (modern, preferred) — returns an iterator object with `__next__`. This is what `for`/`iter()` use if present.
  - **`__getitem__`** only, no `__iter__` (old-style fallback) — Python simulates iteration by calling `obj[0]`, `obj[1]`, `obj[2]`, ... until it hits `IndexError`.
- If a class defines `__iter__`, that's used; `__getitem__` is only consulted as a fallback when `__iter__` is missing.
- An iterator is **exhausted** once it raises `StopIteration` — it can't be rewound or reused. Calling `next()` again just raises `StopIteration` again; to iterate the same data again, you need a fresh iterator (e.g. call `iter()` on the original iterable again).
- **`StopIteration`** is a built-in exception class — a signal, not an error. When `__next__` raises it, that's the standardized way an iterator says "no more values." `for` loops (and other iteration constructs) catch it automatically and just end the loop — you never see a traceback from it in normal use.
- **Common iterables used daily:** `list`, `tuple`, `str`, `dict` (iterates over keys), `set`, `range`, file objects (iterating a file yields it line by line), and `enumerate`/`zip`/`map`/`filter` (which are themselves iterators wrapping other iterables).

In [ ]:
# TODO: Implement a simple iterator class `MyRange` that mimics range(start, end):
# - takes start and end in its constructor
# - implements __iter__ so it can be used in a `for` loop
# - implements __next__ that returns the next value and raises StopIteration when done
# - once exhausted, a given instance should not restart -- iterating it again yields nothing

class MyRange:
    def __init__(self, start, end):
        self._start = start
        self._end = end
        self._index = 0
    
    def __iter__(self):
        return self
    
    def __len__(self):
        return self._end - self._start
    
    def __next__(self):
        a = self._start + self._index
        if a >= self._end:
            raise StopIteration
        self._index += 1
        return a


a = MyRange(4, 6)
while True:
    try:
        print(next(a))
    except StopIteration:
        break

print(len(a))

for i in a: # only possible bc of the itr dunder 
    print(i)

print("========")

for i in a: # only possible bc of the itr dunder 
    print(i)

4
5
2


## Iterables vs Iterators — the protocol

A **protocol** in Python is duck-typing via dunder methods: no inheritance
required. Implement the right dunders and Python's built-in functions/syntax
(`for`, `iter()`, `next()`) treat your object accordingly.

**Iterable** = implements `__iter__` (or falls back to `__getitem__`).
- `__iter__` returns a NEW iterator object (something with `__next__`).
- Its only job: hand back an iterator when `iter()` / `for` asks.
- A list is iterable but NOT its own iterator — `iter(mylist)` gives a fresh
  iterator each time, so you can loop over it repeatedly.

**Iterator** = implements `__iter__` (returning `self`) AND `__next__`.
- `__iter__` returns `self` — it IS already the iterator.
  (Needed so an iterator can be used directly in a `for` loop, since `for`
  calls `iter()` on it first.)
- `__next__` produces the next value, and raises `StopIteration` when
  exhausted — that's how `for` knows to stop.

Summary:
- Iterable's `__iter__` → returns a separate iterator.
- Iterator's `__iter__` → returns self, plus `__next__` does the work.

An iterable is just the data container. It has no `__next__` of its own — it can't walk itself. When you call `iter()` on it, it hands back a separate iterator object that holds `__next__` and the position. So data (the iterable) and iteration (the iterator) are two different objects.

Because of this separation, **the iterable itself never gets exhausted** — only the iterator it hands out does. `mylist` is reusable forever; call `iter(mylist)` again and you get a brand-new, unexhausted iterator. (This only holds for true containers with a real separation, though — a self-iterating object like `MyRange`, where `__iter__` just returns `self`, has no such separation, so it *does* get permanently exhausted once its `__next__` is drained.)

In [ ]:
#example of an iterable:

class MyList:
    def __init__(self, data):
        self.data = data
    def __iter__(self):
        return iter(self.data)   # returns an iterator

#example of an iterator
class MyIterator:
    def __iter__(self):
        return self          # already the iterator
    def __next__(self):
        ...                  # produce next value, or raise StopIteration

Iterable — something you can loop over. It knows how to hand you an iterator when asked. Implements __iter__, which returns a separate iterator object. A list, tuple, dict, string are all iterables.

Iterator — the thing that actually does the looping. It produces values one at a time and remembers where it is. Implements __next__ (give me the next value, or raise StopIteration when done) and __iter__ (returns self, since it's already the iterator).

A regular iterator eventually ends (raises StopIteration once exhausted). A **cyclic iterator** solves this by wrapping around and starting over once it reaches the end, looping the data indefinitely instead of stopping (e.g. `itertools.cycle`).

## Eager vs. lazy evaluation

- **Eager** — compute everything upfront and store it all in memory before using any of it. E.g. `[x**2 for x in range(1000000)]` (list comprehension) builds the full list immediately, even if only the first few values are ever needed.
- **Lazy** — compute each value only when it's actually requested (via `next()`), without holding the rest in memory. E.g. `(x**2 for x in range(1000000))` (generator expression) — nothing is computed until iterated; at any point only the current value + minimal state exists.

Why it matters: for huge or infinite sequences, eager evaluation is either impossible (can't store infinite data) or wasteful (computing/storing values that are never used). Lazy evaluation scales to any size, since it only ever holds "where am I" plus the current value.

**How this connects to iterable/iterator:** those are a *structural* question (who holds `__next__`, is it the same object or a separate one); eager/lazy is a *timing* question (when values get computed). The two axes are independent, but generators collapse both into one default — a generator is always its own iterator (`__iter__` returns `self`) *and* always lazy, since each value is only produced when `next()` resumes its paused execution. That's why generators are "the lazy shortcut for writing iterators": they give you laziness for free, without hand-managing state like `MyRange` does. Cyclic iterators only make sense on the lazy side too — an infinite/repeating sequence could never be stored eagerly.

## Generators — `yield` instead of `return`

A generator is a function that uses `yield` instead of `return`. Calling it doesn't run the body — it immediately returns a **generator object** (which is its own iterator: `__iter__` returns `self`, and it has `__next__`). Each `next()` call resumes execution from wherever it last paused, runs until the next `yield`, hands back that value, and pauses again right there.

**Terminology:** a function that uses `yield` is called a **generator function** — that's the `def` itself. Calling it doesn't run it; it returns a **generator object** — a separate thing, the one you actually call `next()` on. `def my_range(...): ... yield ...` is the generator function; `my_range(4, 6)` is the generator object.

```python
def count_up(n):
    i = 0
    while i < n:
        yield i
        i += 1
```

- First `next()`: runs to `yield i` (`i=0`), pauses, returns `0`.
- Next `next()`: resumes right after that `yield`, does `i += 1`, loops, hits `yield i` again (`i=1`), pauses, returns `1`.
- Once the function falls off the end (loop condition false), Python **automatically raises `StopIteration`** — no manual `raise` needed.

This is the exact same behavior as `MyRange`, but the pause/resume state (`i`, loop position) is handled by Python's function-frame mechanics instead of manually tracked `self._index` — see the eager/lazy note above for why this makes generators "the lazy shortcut for writing iterators."

**Are generators iterators?** Yes. A generator object satisfies the iterator protocol exactly: it has `__iter__` (returns `self`) and `__next__`, both generated automatically by Python from the `yield`-based function. A generator isn't a different *category* from an iterator — it's one convenient way to *produce* one, without writing the class yourself.

**Why use a class-based iterator at all, if generators are so much less code?**
- **A generator only gives you a sequence of values — nothing else.** If the object needs additional methods/attributes beyond iteration (like `MyRange`'s `__len__`, or `Dataset`'s `__getitem__`), a generator object has no place to put them; you need a real class.
- **Generators can't be pickled.** A generator object holds a paused stack frame, which Python can't serialize — this matters for things like multiprocessing, where state sometimes needs to cross process boundaries. Plain instance attributes on a class-based iterator serialize fine.
- **External inspection/reset of iteration state.** With a class, `self._index` is just an attribute — you can read or manipulate it from outside if needed. A generator's internal position is opaque; there's no way to peek at or rewind it.
- **In practice, it's rarely all-or-nothing.** The common real-world pattern is a hybrid: keep the class (for `__len__`, `__getitem__`, other methods, picklability), but implement its `__iter__` method's *body* using `yield` — getting the class's structure and the generator's easy iteration logic at once. This is exactly how `IterableDataset.__iter__` is typically written.

**Are all generators lazy?** Yes, unconditionally — it's not a stylistic choice, it's inherent to how `yield` works. Calling a generator function never executes any code up front, and each value only gets computed when `next()` actually asks for it. If you wanted eager evaluation, you simply wouldn't use a generator (you'd `return` a list instead) — there's no "eager generator."

In [14]:
# TODO: Implement range(start, end) three ways, to compare iterable vs iterator vs generator.
# (MyRange above is already the "iterator" version -- self-iterating, __iter__ returns self.)

# 1. ITERABLE: __iter__ returns a SEPARATE iterator object each call, so this can be looped
#    over multiple times (even nested/concurrently), unlike MyRange.
class MyRangeIterator:
    def __init__(self, start, end):
        self._start = start
        self._end = end
        self._index = 0
    
    def __iter__(self):
        return(self)

    def __next__(self):
        next_ = self._start + self._index
        if next_ >= self._end:
            raise StopIteration
        self._index += 1
        return next_
        

class MyRangeIterable:
    def __init__(self, start, end):
        self._start = start
        self._end  = end

    def __iter__(self):
        return MyRangeIterator(self._start, self._end)


# 2. GENERATOR: a plain function with `yield` -- no classes needed at all.
def my_range(start, end):
    index = 0
    val = start
    while True:
        val = start + index
        if val >= end:
            break
        index += 1
        yield val

# --- test all three ---
print("iterable:")
r = MyRangeIterable(4, 6)
for i in r:
    print(i)
for i in r:              # loop again -- should work, since __iter__ gives a fresh iterator
    print(i)

print("generator:")
for i in my_range(4, 6):
    print(i)
for i in my_range(4, 6): # calling the function again -- a brand-new generator object
    print(i)

print("try next")
a = my_range(4, 6)
print(next(a))
print(next(a))
#print(next(a)) -> error


iterable:
4
5
4
5
generator:
4
5
4
5
try next
4
5


## Why `Dataset` and `DataLoader` are separate

- **`Dataset`** = the data collection. Implements `__len__`/`__getitem__` — random access to any single item by index. No batching or ordering logic.
- **`DataLoader`** = the iteration layer wrapped around it. Implements `__iter__`, handling batching, shuffling, and parallel loading (worker processes), yielding an iterator of batches.

The batching/shuffling/parallelism logic in `DataLoader` is generic — it only needs to call `Dataset.__getitem__(i)` for any index, it doesn't care what the items actually are. So that logic is written once and works for any dataset. If `Dataset` handled its own iteration too, every dataset class would have to reimplement shuffling, batching, and multiprocessing itself. Separating them also means the same `Dataset` can be reused with different `DataLoader` configs (batch size, shuffle on/off, train vs. eval) without touching the data-access logic.

### Summary

- PyTorch has two `Dataset` flavors: **map-style** (`__getitem__`/`__len__`) and **`IterableDataset`** (`__iter__` only). `DataLoader` adapts to whichever one it's given.
- **`__getitem__` is better suited when the data supports true random access** — you know the size up front, and any item can be fetched directly given its index, in O(1) or close to it (e.g. images on disk named/indexed by number, rows in an in-memory array, a database table with a primary key). This is what lets `DataLoader` shuffle freely: shuffling is just generating a random permutation of indices and calling `__getitem__` for each.
- **`__iter__` is better suited when the data can only be consumed sequentially** — its size may be unknown or unbounded, and there's no way to jump to item `N` without first walking through everything before it (e.g. a live stream, a huge text file with variable-length lines, a DB cursor pulling from a query in progress). Here, "shuffle" isn't a free index-permutation trick anymore — it requires buffering a window of items and sampling from that buffer, since you can't randomly seek into the underlying source.

In [20]:
# TODO: Implement the same data (a list of strings, standing in for "lines from a huge file")
# two ways, to compare map-style vs iterable-style Datasets directly.
#
# 1. MAP-STYLE: `MapStyleTextDataset`
#    - takes a list of strings in its constructor
#    - implements __len__ and __getitem__(i) -- random access by index
#    - __getitem__ should return the line UPPERCASED
#
# 2. ITERABLE-STYLE: `StreamingTextDataset`
#    - takes a list of strings in its constructor
#    - implements __iter__ using a generator (yield) -- NOT __getitem__/__len__, since the
#      whole point is sequential-only access, no random access by index
#    - __iter__ should yield each line UPPERCASED
#
# Then test both:
# - MapStyleTextDataset: try dataset[2] directly, and len(dataset)
# - StreamingTextDataset: loop over an instance with a `for` loop, TWICE -- should work both
#   times and yield everything again, since __iter__ (a generator function) creates a
#   brand-new generator object each call. Then try dataset[2] -- should fail, no __getitem__.

class MapStyleTextDataset:
    def __init__(self, lines):
        self._lines = lines
        self._length = len(self._lines)

    def __len__(self):
        return self._length

    def __getitem__(self, i):
        if i >= self._length:
            raise ValueError("index out of bound")
        else:
            return self._lines[i].upper()


class StreamingTextDataset:
    def __init__(self, lines):
        self._lines = lines
        self._length = len(self._lines)

    def __iter__(self):
        index = 0
        while True:
            if index >= self._length:
                break
            else:
                yield self._lines[index].upper()
                index += 1

l = ['sfh', 'kuy', 'tre'] 
map_l = MapStyleTextDataset(l)
print(map_l[0])
itr_l = StreamingTextDataset(l)
print("++++++++++")
for m in itr_l:
    print(m)
print("++++++++++")
print(next(iter(itr_l)))
print("++++++++++")
for m in itr_l:
    print(m)


SFH
++++++++++
SFH
KUY
TRE
++++++++++
SFH
++++++++++
SFH
KUY
TRE


## "iterables don't exhaust" only holds for the iterable itself

```python
itr_l = iter(StreamingTextDataset(l))
for m in itr_l:
    print(m)
```

This *does* exhaust after one loop — and that's correct, not a contradiction. `iter()` calls `__iter__`, and since it's a generator function, that call immediately returns the **generator object** (the iterator) — `itr_l` holds the iterator, not the iterable. Iterators are always single-use; that part hasn't changed.

The `StreamingTextDataset` *instance* is the iterable, and it's still reusable — but only if you keep a reference to it and let each loop create its own fresh iterator internally:

```python
ds = StreamingTextDataset(l)
for m in ds:      # calls iter(ds) internally -> fresh generator
    print(m)
for m in ds:      # calls iter(ds) again -> another fresh generator
    print(m)
```

This works both times. The difference: calling `iter()` yourself and saving the result peels off the (single-use) iterator; looping over the object directly, repeatedly, lets `__iter__` mint a new iterator each time.